In [1]:
import pandas as pd

In [ ]:
# use of .transform() method 
# Usually used after groupby. This works similar to .apply() but does not shrink the dataframe for the single final value
# instead it results in a Series of same length
import pandas as pd


df = pd.DataFrame({
    'Category': ['A', 'A', 'B', 'B'],
    'Sales': [100, 200, 300, 100]
})

# Use transform to get the sum per category, but keep the 4-row shape
df['Category_Total'] = df.groupby('Category')['Sales'].transform('sum')

# Now you can do row-level math easily
df['Percent_of_Total'] = df['Sales'] / df['Category_Total']
df.groupby('Category')['Sales']
# USE CASE 
# TO fill the null values of some column based on the specific category it belongs to
df.groupby('Category')['Review_Rating'].transform(lambda x: x.fillna(x.median()))

# this fills the null values by the median rating category wise 

# Forming ROI Column

In [ ]:
import pandas as pd

# 1. Filter for Non-Zero financials and English language
clean_df = df[(df['budget'] > 0) & (df['revenue'] > 0) & (df['original_language'] == 'en')].copy()

# 2. Create the ROI column (The Business Metric)
clean_df['roi'] = (clean_df['revenue'] - clean_df['budget']) / clean_df['budget']

# 3. Perform Stratified Sampling (optional but recommended)
# This takes 5000 movies while keeping the genre proportions intact
sample_df = clean_df.groupby('genres', group_keys=False).apply(lambda x: x.sample(frac=0.5)) 
# Or use a fixed number:
sample_df = clean_df.sample(n=min(5000, len(clean_df)), random_state=42)

print(f"Final sample size: {len(sample_df)}")

# Why we need to change the dtype as per our requirements?

`The mystery is likely solved: Since your director column has the str dtype (the new specialized StringDtype introduced in recent Pandas versions), it is much stricter than the old object dtype.`

`When you run .str.split(), the output is a Python List. The str dtype cannot hold lists; it can only hold strings. When you try to save those lists back into a column locked as str, Pandas can't reconcile the types and forces them to NaN.`


`You need to convert the column to object type first so it has the flexibility to hold the lists generated by the split.`

# Forming New columns using the inflation metric 

In [ ]:
import cpi
import pandas as pd

cpi.update()
target_year = 2025

unique_years = second_df['year_of_release'].dropna().unique()

inflation_factors = {
    year: cpi.inflate(1, year, to=target_year)
    for year in unique_years
}
second_df['inflation_factor'] = second_df['year_of_release'].map(inflation_factors)
second_df['budget_adj'] = second_df['budget'] * second_df['inflation_factor']
second_df['revenue_adj'] = second_df['revenue'] * second_df['inflation_factor']